# Spatial Analysis with Squidpy

This notebook performs comprehensive spatial analysis:
- Spatial neighborhood analysis
- Cell-cell interaction analysis  
- Spatial clustering
- Niche identification
- Spatial statistics

**Input:** Annotated AnnData from 02_phenotyping.ipynb

**Output:** Spatial metrics and interaction networks

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sc.settings.set_figure_params(dpi=80, facecolor='white')
print(f"Squidpy version: {sq.__version__}")

In [ ]:
# Load annotated data
DATA_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../data/processed")
FIGURES_DIR = Path("../figures/03_spatial_analysis")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_NAME = "xenium_sample_01"
adata = sc.read_h5ad(DATA_DIR / f"{SAMPLE_NAME}_annotated.h5ad")

print(f"Data shape: {adata.shape}")
print(f"Cell types: {adata.obs['celltype'].nunique()}")

## 1. Build Spatial Graph

In [ ]:
# Build spatial neighbor graph
sq.gr.spatial_neighbors(adata, coord_type='generic', n_neighs=10)
print("Spatial graph constructed")

# Visualize spatial connectivity
if 'spatial' in adata.obsm:
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    sq.pl.spatial_scatter(
        adata,
        color='celltype',
        size=2,
        connectivity_key='spatial_connectivities',
        edges_width=0.2,
        edges_color='gray',
        ax=ax
    )
    plt.savefig(FIGURES_DIR / 'spatial_connectivity.png', dpi=300, bbox_inches='tight')
    plt.show()

## 2. Neighborhood Enrichment Analysis

In [ ]:
# Calculate neighborhood enrichment
sq.gr.nhood_enrichment(adata, cluster_key='celltype')

# Plot enrichment
sq.pl.nhood_enrichment(
    adata,
    cluster_key='celltype',
    method='ward',
    cmap='coolwarm',
    vmin=-50,
    vmax=50
)
plt.savefig(FIGURES_DIR / 'neighborhood_enrichment.png', dpi=300, bbox_inches='tight')
plt.show()

print("Neighborhood enrichment calculated")

## 3. Co-occurrence Analysis

In [ ]:
# Calculate co-occurrence probability
sq.gr.co_occurrence(
    adata,
    cluster_key='celltype'
)

# Plot co-occurrence
sq.pl.co_occurrence(
    adata,
    cluster_key='celltype',
    clusters=None,  # Plot all cell types
    figsize=(8, 8)
)
plt.savefig(FIGURES_DIR / 'co_occurrence.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Spatial Autocorrelation

In [ ]:
# Find marker genes if not already computed
if 'rank_genes_leiden' not in adata.uns:
    sc.tl.rank_genes_groups(adata, groupby='celltype', method='wilcoxon')

# Get top genes for spatial autocorrelation
genes_to_test = adata.var_names[:100]  # Top 100 genes by expression

# Calculate Moran's I
sq.gr.spatial_autocorr(
    adata,
    mode='moran',
    genes=genes_to_test,
    n_jobs=1
)

# Plot top spatially variable genes
top_spatially_variable = adata.uns['moranI'].head(20)
print("\nTop 20 spatially variable genes:")
print(top_spatially_variable[['I', 'pval_norm']])

## 5. Spatial Clustering and Domains

In [ ]:
# Calculate spatial domains using clustering on spatial graph
from sklearn.cluster import KMeans

# Use spatial coordinates for domain detection
if 'spatial' in adata.obsm:
    coords = adata.obsm['spatial']
    
    # K-means clustering on spatial coordinates
    n_domains = 5
    kmeans = KMeans(n_clusters=n_domains, random_state=42)
    adata.obs['spatial_domain'] = kmeans.fit_predict(coords).astype(str)
    
    # Plot spatial domains
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    sq.pl.spatial_scatter(
        adata,
        color='spatial_domain',
        size=2,
        ax=axes[0],
        title='Spatial Domains'
    )
    
    sq.pl.spatial_scatter(
        adata,
        color='celltype',
        size=2,
        ax=axes[1],
        title='Cell Types'
    )
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'spatial_domains.png', dpi=300, bbox_inches='tight')
    plt.show()

## 6. Ligand-Receptor Interaction Analysis

In [ ]:
# Perform ligand-receptor interaction analysis
sq.gr.ligrec(
    adata,
    n_perms=100,
    cluster_key='celltype',
    copy=False,
    use_raw=False
)

print("Ligand-receptor analysis completed")

# Access results
if 'ligrec' in adata.uns:
    lr_results = adata.uns['ligrec']
    print(f"\nNumber of significant interactions: {len(lr_results)}")

## 7. Ripley's Statistics

In [ ]:
# Calculate Ripley's statistics for spatial patterns
mode = 'L'  # L-function (normalized K-function)

sq.gr.ripley(
    adata,
    cluster_key='celltype',
    mode=mode,
    max_dist=500
)

# Plot Ripley's statistics
cell_types_to_plot = adata.obs['celltype'].value_counts().head(5).index.tolist()

sq.pl.ripley(
    adata,
    cluster_key='celltype',
    mode=mode,
    clusters=cell_types_to_plot
)
plt.savefig(FIGURES_DIR / 'ripley_statistics.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Save Results

In [ ]:
# Save data with spatial analysis results
output_file = OUTPUT_DIR / f"{SAMPLE_NAME}_spatial_analysis.h5ad"
adata.write_h5ad(output_file)

print(f"\nSpatial analysis results saved to: {output_file}")

# Export neighborhood enrichment scores
if 'celltype_nhood_enrichment' in adata.uns:
    nhood_df = pd.DataFrame(
        adata.uns['celltype_nhood_enrichment']['zscore'],
        index=adata.uns['celltype_nhood_enrichment']['names'],
        columns=adata.uns['celltype_nhood_enrichment']['names']
    )
    nhood_df.to_csv(OUTPUT_DIR / f"{SAMPLE_NAME}_neighborhood_enrichment.csv")
    print(f"Neighborhood enrichment exported")

# Export Moran's I results
if 'moranI' in adata.uns:
    adata.uns['moranI'].to_csv(OUTPUT_DIR / f"{SAMPLE_NAME}_morans_i.csv")
    print(f"Moran's I statistics exported")